# `indic/01` — Fetch IndicMT Eval & Verify Corpus

**Purpose:** Download the IndicMT Eval dataset from Hugging Face, verify the
exact corpus dimensions required by the paper (1,400 segments × 5 Indic languages
= 7,000 rows total), inspect MQM severity-bucket counts, and save one consolidated
CSV per language to `data/processed/` for use by all downstream notebooks.

**Outputs produced:**
```
data/processed/gujarati_indicmt.csv   (1 400 rows)
data/processed/hindi_indicmt.csv      (1 400 rows)
data/processed/malayalam_indicmt.csv  (1 400 rows)
data/processed/marathi_indicmt.csv    (1 400 rows)
data/processed/tamil_indicmt.csv      (1 400 rows)
```

**Paper reference:** §3 Experimental Setup — IndicMT Eval (Sai et al., 2023),
5 ENG→Indic directions, 1,400 MQM-annotated segment pairs per language,
labelled across 11 error types and 3 severity levels.

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["datasets", "pandas"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}..."); _install(pkg)

print("Dependencies ready.")

## Configuration

All paths are relative to the repository root.  
Set `DATA_DIR` to wherever you want the per-language CSVs written.

In [ ]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR = Path("../../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset identifier on Hugging Face ──────────────────────────────────────────
HF_DATASET = "ai4bharat/IndicMT-Eval"

# ── Language keys used for filenames (lowercase full names) ───────────────────
# Matches the pattern expected by notebooks 03-12: <language>_indicmt.csv
LANG_CONFIGS = {
    "gujarati":  "en-gu",
    "hindi":     "en-hi",
    "malayalam": "en-ml",
    "marathi":   "en-mr",
    "tamil":     "en-ta",
}

EXPECTED_ROWS      = 1_400
EXPECTED_LANGUAGES = 5

print(f"Output directory : {DATA_DIR.resolve()}")
print(f"Languages        : {list(LANG_CONFIGS.keys())}")

## Step 1 — Load IndicMT Eval from Hugging Face

In [ ]:
from datasets import load_dataset
import pandas as pd

raw = {}

for lang, config in LANG_CONFIGS.items():
    ds = load_dataset(HF_DATASET, config, split="test", trust_remote_code=True)
    df = ds.to_pandas()
    raw[lang] = df
    print(f"  {lang} ({config})  {len(df):,} rows  |  columns: {list(df.columns)}")

print(f"\nTotal rows loaded: {sum(len(v) for v in raw.values()):,}")

## Step 2 — Normalise Column Names

Rename HuggingFace raw columns to the canonical schema used by notebooks 03–12:

| Raw HF column | Canonical name | Description |
|---|---|---|
| `src` / `source` | `Source` | English source sentence |
| `hyp` / `mt` / `translation` | `Translation` | MT hypothesis (native script) |
| `ref` / `reference` | `Reference` | Human reference translation |
| `mqm_score` / `score` / `z_mean` | `DA_score` | Human DA/MQM score |
| severity field | `Error_Severity` | Severity bucket (Def/VL/L/M/H/VH) |

In [ ]:
import numpy as np

SRC_ALIASES = ["src", "source", "en", "english", "sentence1"]
HYP_ALIASES = ["hyp", "hypothesis", "mt", "target", "sentence2", "translation"]
REF_ALIASES = ["ref", "reference", "ref_a", "refa", "human_ref"]
MQM_ALIASES = ["DA_score", "mqm_score", "mqm", "score", "human_score", "annotation", "z_mean"]

def first_match(df, aliases):
    for a in aliases:
        if a in df.columns:
            return a
    return None

def mqm_to_severity(score):
    if pd.isna(score) or score == 0: return "Def"
    elif score <= 1:  return "VL"
    elif score <= 5:  return "L"
    elif score <= 10: return "M"
    elif score <= 25: return "H"
    else:             return "VH"

normalised = {}

for lang, df in raw.items():
    d = df.copy()
    src_c = first_match(d, SRC_ALIASES)
    hyp_c = first_match(d, HYP_ALIASES)
    ref_c = first_match(d, REF_ALIASES)
    mqm_c = first_match(d, MQM_ALIASES)
    d.rename(columns={
        src_c: "Source",
        hyp_c: "Translation",
        ref_c: "Reference",
        mqm_c: "DA_score",
    }, inplace=True)
    d["Error_Severity"] = d["DA_score"].apply(mqm_to_severity)
    d["lang"] = lang  # lowercase full name e.g. 'gujarati'
    normalised[lang] = d
    print(f"  {lang}: Source/Translation/Reference/DA_score/Error_Severity all present: ",
          all(c in d.columns for c in ['Source','Translation','Reference','DA_score','Error_Severity']))

## Step 3 — Save Per-Language CSVs

Each file saved to `data/processed/<language>_indicmt.csv`.
All downstream notebooks (03–12) read from this directory.

In [ ]:
saved_paths = {}

for lang, df in normalised.items():
    out_path = DATA_DIR / f"{lang}_indicmt.csv"
    df.to_csv(out_path, index=False)
    saved_paths[lang] = out_path
    print(f"  Saved  {out_path}  ({len(df):,} rows)")

print(f"\n  {len(saved_paths)} files written to {DATA_DIR.resolve()}")

## Step 4 — Summary

In [ ]:
print('=' * 60)
print('IndicMT Eval -- corpus summary')
print('=' * 60)
print(f"{'Lang':<12}  {'Rows':>6}  {'Source':>8}  {'DA_score':>10}  {'Error_Severity':>14}")
print('-' * 60)
for lang, df in normalised.items():
    src_c = 'Source'         if 'Source'         in df.columns else '--'
    mqm_c = 'DA_score'       if 'DA_score'       in df.columns else '--'
    sev_c = 'Error_Severity' if 'Error_Severity' in df.columns else '--'
    print(f'  {lang:<10}  {len(df):>5,}    {src_c:>8}  {mqm_c:>10}  {sev_c}')
grand_total = sum(len(v) for v in normalised.values())
print('-' * 60)
print(f'  TOTAL       {grand_total:>5,}')